<a href="https://colab.research.google.com/github/zeegy99/ML-notebooks/blob/main/DDPM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Implementation of DDPM Paper. A writeup of the math is linked [Here](https://www.overleaf.com/read/pszbcvbmptms#9f695c)


---





In [2]:
import torch
import csv
from PIL import Image, ImageOps, ImageFilter
import random
from torchvision import transforms

beta = 0.001


def vector_to_image(tensor):
  transform = transforms.Compose([
      transforms.Normalize((0,), (2,)),
      transforms.Normalize((-0.5,), (1,)),
      transforms.ToPILImage()
  ])

  return transform(tensor)
def forward_process(x_0, t):

#   transform = transforms.Compose([
#     transforms.ToTensor(),
#     transforms.Normalize((0.5,), (0.5,))
# ])
  # x_0 = transform(x_0)
  alpha = 1 - beta
  alpha_bar = alpha ** t

  eps = torch.randn(x_0.shape)
  x_t = alpha_bar ** .5 * x_0 + (1-alpha_bar) ** .5  * eps

  return x_t, eps

def forward_process_not_preprocessed(x_0, t):

  transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])
  x_0 = transform(x_0)
  alpha = 1 - beta
  alpha_bar = alpha ** t

  eps = torch.randn(x_0.shape)
  x_t = alpha_bar ** .5 * x_0 + (1-alpha_bar) ** .5  * eps

  return x_t, eps

def backward_process(model, T, shape):
    x_t = torch.randn(3, shape[0], shape[1])

    for t in reversed(range(T)):
        eps_pred = model(x_t, t)


        alpha = 1 - beta
        alpha_bar = alpha ** t

        #from paper
        mu = 1 / alpha ** 0.5 * (x_t - (1- alpha)/((1 - alpha_bar) ** .5) * eps_pred)

        #adding back some noise
        z = torch.randn_like(x_t) if t > 0 else 0
        x_t = mu + beta**0.5 * z

    return x_t



In [24]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""
! pip install open_clip_torch matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.5 MB/s eta 0:00:00


In [ ]:


print(model.parameters())

RuntimeError: Failed to import diffusers.models.unets.unet_2d because of the following error (look up to see its traceback):
Failed to import diffusers.loaders.single_file_model because of the following error (look up to see its traceback):
Could not import module 'AutoImageProcessor'. Are this object's requirements defined correctly?

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from diffusers import UNet2DModel

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

dataset = datasets.CIFAR10(
    root='./data',
    train=True,
    download=False,
    transform=transform
)

dataloader = DataLoader(
    dataset,
    batch_size=64,
    shuffle=True
)

model = UNet2DModel(
    sample_size=32,
    in_channels=3,
    out_channels=3,
    block_out_channels=(64, 128, 256, 512),
)


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [ ]:
from torch.optim import Adam
import torch.nn.functional as F
optimizer = Adam(model.parameters(), lr=1e-4)
num_epochs = 10
T = 1000
for epoch in range(num_epochs):
    for x_0, labels in dataloader:

        t = random.randint(1, T)
        x_t, eps = forward_process(x_0, t)

        eps_pred = model(x_t, t).sample

        loss = F.mse_loss(eps, eps_pred)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()